In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))


In [2]:
refine_plus_export_pool = Path("refine_plus_export_pool")
export_folder = Path("combined_db")

In [3]:
import os
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools

lfs_to_join: list[LazyFrame] = [
    pl.scan_parquet(refine_plus_export_pool / table_name)
    for table_name in os.listdir(refine_plus_export_pool)
] + [
    dp.combined_atlas,
    dp.combined_mask.unique(
    subset=["i", "j", "face"], keep="first" # Sometimes duplicates can occur but are rare
    ).rename({
        "lod_level": "detection_lod_level",
        "lod_code": "detection_lod_code",
        "row_id" : "boulder_id"
    })
]

for lf in lfs_to_join:
    print(lf.collect_schema().names())

['i', 'j', 'face', 'area']
['i', 'j', 'face', 'detailed_survey band depth 350', 'detailed_survey band depth 440', 'detailed_survey slope 1000', 'detailed_survey ratio 1000', 'detailed_survey sigma band depth 350', 'detailed_survey sigma band depth 440', 'detailed_survey sigma slope 1000', 'detailed_survey sigma ratio 1000']
['i', 'j', 'face', 'recona band depth 350', 'recona band depth 440', 'recona slope 1000', 'recona ratio 1000', 'recona sigma band depth 350', 'recona sigma band depth 440', 'recona sigma slope 1000', 'recona sigma ratio 1000']
['i', 'j', 'face', 'reconb band depth 350', 'reconb band depth 440', 'reconb slope 1000', 'reconb ratio 1000', 'reconb sigma band depth 350', 'reconb sigma band depth 440', 'reconb sigma slope 1000', 'reconb sigma ratio 1000']
['i', 'j', 'face', 'detailed_survey band depth', 'detailed_survey reflectance', 'detailed_survey slope1 poly', 'detailed_survey slope2 poly', 'detailed_survey sigma band depth', 'detailed_survey sigma reflectance', 'deta

c:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
ChunkingTools.join_in_chunks(
    export_folder = export_folder,
    lfs_to_join = lfs_to_join,
    chunks = QCubeChunk.generate(depth=4),
    n_jobs = 8,
)

My calculation:   0%|          | 0/1536 [00:00<?, ?it/s]

c:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [20]:
# pl.scan_parquet(export_folder).group_by("boulder_id").agg().collect()

pl.scan_parquet(export_folder).filter(pl.col("boulder_id").is_in(pl.Series([1318804, 2838414]).implode())).collect()

i,j,face,area,detailed_survey band depth 350,detailed_survey band depth 440,detailed_survey slope 1000,detailed_survey ratio 1000,detailed_survey sigma band depth 350,detailed_survey sigma band depth 440,detailed_survey sigma slope 1000,detailed_survey sigma ratio 1000,recona band depth 350,recona band depth 440,recona slope 1000,recona ratio 1000,recona sigma band depth 350,recona sigma band depth 440,recona sigma slope 1000,recona sigma ratio 1000,reconb band depth 350,reconb band depth 440,reconb slope 1000,reconb ratio 1000,reconb sigma band depth 350,reconb sigma band depth 440,reconb sigma slope 1000,reconb sigma ratio 1000,detailed_survey band depth,detailed_survey reflectance,detailed_survey slope1 poly,detailed_survey slope2 poly,detailed_survey sigma band depth,detailed_survey sigma reflectance,detailed_survey sigma slope1 poly,detailed_survey sigma slope2 poly,recona band depth,recona reflectance,recona slope1 poly,recona slope2 poly,recona sigma band depth,recona sigma reflectance,recona sigma slope1 poly,recona sigma slope2 poly,reconc band depth,reconc reflectance,reconc slope1 poly,reconc slope2 poly,reconc sigma band depth,reconc sigma reflectance,reconc sigma slope1 poly,reconc sigma slope2 poly,uint8_reflectance,32bit_reflectance,positions_x,positions_y,positions_z,detection_lod_level,detection_lod_code,boulder_id
u32,u32,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,u8,f32,f32,f32,f32,u8,str,u32
532,7729,"""posx""",0.000864,1.002937,1.014572,0.997814,NaN,0.00194,0.002496,0.001783,NaN,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,0.010652,0.133656,-0.15063,-0.13109,4,"""BBBD""",1318804
532,7730,"""posx""",0.000825,1.002937,1.014572,0.997814,NaN,0.00194,0.002496,0.001783,NaN,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98,0.016912,0.133681,-0.150621,-0.131083,4,"""BBBD""",1318804
532,7731,"""posx""",0.000831,1.002937,1.014572,0.997814,NaN,0.00194,0.002496,0.001783,NaN,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110,0.019118,0.133705,-0.150614,-0.131075,4,"""BBBD""",1318804
532,7732,"""posx""",0.00092,1.002937,1.014572,0.997814,NaN,0.00194,0.002496,0.001783,NaN,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,111,0.019199,0.133732,-0.150605,-0.131067,4,"""BBBD""",1318804
532,7733,"""posx""",0.000973,1.002937,1.014572,0.997814,NaN,0.00194,0.002496,0.001783,NaN,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,108,0.018697,0.133762,-0.150594,-0.131059,4,"""BBBD""",1318804
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8021,2792,"""posz""",0.000999,1.002876,1.014286,0.996977,NaN,0.001521,0.000791,0.00068,NaN,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,0.14115,0.040202,-0.003974,-0.005293,0.011021,0.000158,0.000072,0.000027,null,null,null,null,null,null,null,null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64,0.01103,-0.156207,0.049694,0.149672,4,"""CDCD""",2838414
8021,2793,"""posz""",0.001036,1.002876,1.014286,0.996977,NaN,0.001521,0.000791,0

In [6]:
# from boulder_statistics.analysis.data_product_encyclopedia import FACES

# df = df.with_columns(
#     pl.col("face").cast(pl.Enum(FACES))
# ).select(
#     pl.col("face"),
#     pl.col("i"),
#     pl.col("j"),
#     pl.col("uint8_reflectance").alias("u8_R"),
#     pl.col("32bit_reflectance").alias("32f_R"),
#     pl.col("positions_x").alias("x"),
#     pl.col("positions_y").alias("y"),
#     pl.col("positions_z").alias("z"),
#     pl.col("area")
# ).with_columns(
#     (pl.col("x") ** 2 + pl.col("y") ** 2 + pl.col("z") ** 2).sqrt().alias("r")
# ).with_columns(
#     x_hat = pl.col("x") / pl.col("r"),
#     y_hat = pl.col("y") / pl.col("r"),
#     z_hat = pl.col("z") / pl.col("r"),
# ).with_columns(
#     theta = pl.col("z_hat").arccos(),
#     phi = pl.arctan2(pl.col("y_hat"), pl.col("x_hat"))
# )

# df.sink_parquet(clean_db_export_path, engine = "streaming")

In [7]:
pl.scan_parquet(clean_db_export_path).filter(pl.col("i") == 2).collect()

NameError: name 'clean_db_export_path' is not defined